# Synthetic Dataset Generation
# Part I b: Unstructured Reports with missing values

In [1]:
from collections import OrderedDict, defaultdict
from pathlib import Path
import copy
import json
import random


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        notebook = directory / "code_to_obtain_SR_data" / "diversify_balanced_synthetic_SR.ipynb"
        if notebook.exists():
            return directory
    raise FileNotFoundError("Could not locate the synthetic_dataset project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "code_to_obtain_SR_data" / "jsonl_data"
BALANCED_DATA_PATH = DATA_DIR / "balanced_synthetic_SR.jsonl"
PROCESSED_DATA_PATH = DATA_DIR / "balanced_synthetic_SR_processed.jsonl"
MODIFIED_DATA_PATH = DATA_DIR / "balanced_synthetic_SR_modified.jsonl"

PROCESSING_SEED = 44
UNIT_CONVERSION_SEED = 45


### We will introduce some distortions in the balanced dataset to mimic missing information and differences in reporting measurements (e.g. cm vs mm)

In [2]:
with BALANCED_DATA_PATH.open(encoding="utf-8") as f:
    for index, line in enumerate(f):
        if index >= 10:
            break
        print(json.dumps(json.loads(line), indent=2))
        print("\n" + "=" * 50 + "\n")


{
  "local_tumor_status": {
    "morphology": "polypoid",
    "mucinous_component": "no",
    "circumferential_tumor_involvement_start": 5,
    "circumferential_tumor_involvement_end": 7,
    "rectal_tumor_location_in_rectum": "low",
    "distance_from_anorectal_junction_to_distal_tumor_border_cm": 1.1,
    "relation_to_anterior_peritoneal_reflection": "below",
    "tumor_length_cm": 5.1,
    "t_stage": "T3b",
    "invaded_structures": "",
    "sphincter_complex_invasion": "no",
    "lowest_part_of_sphincter_invasion": "not applicable"
  },
  "mesorectal_fascia_involement": {
    "shortest_distance_to_MRF_mm": 6.2,
    "margin": "clear(> 1 mm)",
    "circumferential_location_of_shortest_distance": null,
    "involved_by": "not applicable"
  },
  "lymph_nodes_and_tumor_deposits": {
    "regional_total_suspicious_mesorectal_nodes": 1,
    "nodes_short_axis_ge_9mm": 0,
    "nodes_from_5_to_8mm_with_2_morphologic_criteria": 1,
    "nodes_5mm_with_all_3_morphologic_criteria": 0,
    "mucino

In [3]:
def get_value_distributions(outputs_list):
    """
    Calculates the distribution of values for all fields, including nested fields,
    across a list of dictionary outputs.

    Args:
        outputs_list (list): A list of parsed dictionary records.

    Returns:
        dict: A dictionary where keys are field paths (e.g., "local_tumor_status.morphology")
              and values are dictionaries containing value counts for that field.
    """
    distributions = defaultdict(lambda: defaultdict(int))

    def collect_values(data, path=""):
        if isinstance(data, dict):
            for key, value in data.items():
                new_path = f"{path}.{key}" if path else key
                collect_values(value, new_path)
        elif isinstance(data, list):
            distributions[path][str(data)] += 1
        else:
            distributions[path][str(data)] += 1

    for parsed_output in outputs_list:
        collect_values(parsed_output)

    return distributions

all_outputs = []

try:
    with BALANCED_DATA_PATH.open(encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                all_outputs.append(json.loads(line.strip()))
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON on line {line_num}: {line.strip()} - {e}")
                continue
except FileNotFoundError:
    print(f"Error: The file {BALANCED_DATA_PATH} was not found.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred while reading the file: {e}")
    exit()

print(f"Successfully loaded {len(all_outputs)} records from {BALANCED_DATA_PATH}")

distributions = get_value_distributions(all_outputs)

print("\n" + "="*50)
print("Value Distributions Across All Fields:")
print("="*50 + "\n")
for field_path, value_counts in sorted(distributions.items()):
    print(f"Field: {field_path}")
    total_occurrences = sum(value_counts.values())
    for value, count in value_counts.items():
        percentage = (count / total_occurrences) * 100
        print(f"  '{value}': {count} occurrences ({percentage:.2f}%)")
    print("\n")
    print("-------------------------------------------------------------------")

Successfully loaded 1000 records from C:\Users\simmu\Desktop\Paper\synthetic_dataset\code_to_obtain_SR_data\jsonl_data\balanced_synthetic_SR.jsonl

Value Distributions Across All Fields:

Field: emvi.emvi_clock_position
  'None': 538 occurrences (53.80%)
  '2': 34 occurrences (3.40%)
  '3': 38 occurrences (3.80%)
  '8': 32 occurrences (3.20%)
  '10': 46 occurrences (4.60%)
  '11': 51 occurrences (5.10%)
  '7': 30 occurrences (3.00%)
  '5': 38 occurrences (3.80%)
  '4': 44 occurrences (4.40%)
  '9': 23 occurrences (2.30%)
  '6': 39 occurrences (3.90%)
  '1': 48 occurrences (4.80%)
  '12': 39 occurrences (3.90%)


-------------------------------------------------------------------
Field: emvi.presence_of_emvi
  'no': 538 occurrences (53.80%)
  'yes': 462 occurrences (46.20%)


-------------------------------------------------------------------
Field: local_tumor_status.circumferential_tumor_involvement_end
  '7': 63 occurrences (6.30%)
  '5': 84 occurrences (8.40%)
  '2': 77 occurrences 

### Script to modify the dataset

**Local Tumor Status Fields**\
**morphology**
 - 5% replaced with "missing"

**mucinous_component**
 - 1% with "missing"\
Lowest missing rate, suggesting this is usually well-documented

**circumferential_tumor_involvement_start AND end**
 - 3% null
Both fields set together

**distance_from_anorectal_junction_to_distal_tumor_border_cm**
- 3% null

**relation_to_anterior_peritoneal_reflection**
- 3% "missing"

**tumor_length_cm**
 - 3% null

**t_stage**
- 2% "missing"
Low missing rate overall

**Conditional Logic for Tumor Invasion**
**invaded_structures**
 - 3% "missing" ONLY if t_stage "T4b" AND rectal_tumor_location_in_rectum NOT "low"
Only relevant for advanced tumors (T4b) that aren't in low rectum


**sphincter_complex_invasion**
 - 3% ONLY if rectal_tumor_location_in_rectum is "low"\
Only relevant for low rectal tumors where sphincter involvement is a concern\
Clinical logic: sphincter invasion only matters for tumors near the anal sphincter

**lowest_part_of_sphincter_invasion**
 - 5% ONLY if sphincter_complex_invasion IS NOT "no"\
Only processed when there IS sphincter invasion\
Clinical logic: can't specify which part is invaded if there's no invasion at all

**Mesorectal Fascia Fields with Cascading Logic**\
**margin**
 - 3% "missing" ONLY if there is value (not empty/null)\
Won't make already-empty fields "missing"\
Cascading rule: When margin becomes "missing", automatically sets:\
circumferential_location_of_shortest_distance -> null\
involved_by -> "missing"\
missing  margin status -> missing related details

**circumferential_location_of_shortest_distance**
 - 3% null ONLY if margin is "involved(<= 1 mm)"\
Only relevant when margin is actually involved\
Won't affect records already processed by the margin cascading rule

**involved_by**
 - 3% "missing" ONLY if NOT "not applicable" AND NOT already "missing"\
Excludes records where it's not applicable or already processed by cascading rule\
Only relevant when there's actual involvement to characterize

**Lymph Node Data Cleanup**\
**Zero-to-null conversion**
- for all lymph node count fields\
Radiologists rarely state that they see zero lymph nodes of particular type

**extramesorectal_location field removal**
 - Remove, unnecessary field

**EMVI Fields with Cascading Logic**\
**presence_of_emvi**
 - 3% "missing"
Cascading rule: When set to "missing", automatically sets emvi_clock_position -> null\
no EMVI presence -> no location

**emvi_clock_position**
 - 5% null ONLY if presence_of_emvi is "yes"\
Only relevant when EMVI is actually present\
Won't affect records already processed by the presence_of_emvi cascading rule\
Higher missing rate (5%) suggests clock position is harder to determine even when EMVI is present

**n_stage**
 - 3% with "missing"\
Straightforward percentage-based replacement for nodal staging


In [4]:
def process_medical_data(data_list, seed=PROCESSING_SEED):
    """
    Process a list of 1000 medical records and introduce missing values according to specified rules.

    Args:
        data_list: List of 1000 dictionaries with medical record structure

    Returns:
        List of processed dictionaries with missing values introduced
    """

    processed_data = copy.deepcopy(data_list)
    total_records = len(processed_data)
    rng = random.Random(seed)

    if not processed_data:
        return processed_data
    required_mrf_fields = {
        "shortest_distance_to_MRF_mm",
        "margin",
        "circumferential_location_of_shortest_distance",
        "involved_by",
    }
    actual_mrf_fields = set(processed_data[0].get("mesorectal_fascia_involement", {}))
    missing_fields = required_mrf_fields - actual_mrf_fields
    if missing_fields:
        raise KeyError(f"Missing MRF fields in input schema: {sorted(missing_fields)}")

    # get random indices
    def get_random_indices(percentage, total):
        count = int(total * percentage / 100)
        return rng.sample(range(total), count)

    indices = get_random_indices(5, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["morphology"] = "missing"

    indices = get_random_indices(1, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["mucinous_component"] = "missing"

    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["circumferential_tumor_involvement_start"] = None
        processed_data[i]["local_tumor_status"]["circumferential_tumor_involvement_end"] = None

    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["distance_from_anorectal_junction_to_distal_tumor_border_cm"] = None

    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["relation_to_anterior_peritoneal_reflection"] = "missing"

    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["tumor_length_cm"] = None


    indices = get_random_indices(2, total_records)
    for i in indices:
        processed_data[i]["local_tumor_status"]["t_stage"] = "missing"

    eligible_indices = []
    for i, record in enumerate(processed_data):
        t_stage = record["local_tumor_status"]["t_stage"]
        location = record["local_tumor_status"]["rectal_tumor_location_in_rectum"]
        if t_stage == "T4b" and location != "low":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.03))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["local_tumor_status"]["invaded_structures"] = "missing"


    eligible_indices = []
    for i, record in enumerate(processed_data):
        location = record["local_tumor_status"]["rectal_tumor_location_in_rectum"]
        if location == "low":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.03))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["local_tumor_status"]["sphincter_complex_invasion"] = "missing"


    eligible_indices = []
    for i, record in enumerate(processed_data):
        sphincter_invasion = record["local_tumor_status"]["sphincter_complex_invasion"]
        if sphincter_invasion != "no":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.05))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["local_tumor_status"]["lowest_part_of_sphincter_invasion"] = "missing"

    eligible_indices = []
    for i, record in enumerate(processed_data):
        margin = record["mesorectal_fascia_involement"]["margin"]
        if margin and margin != "":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.03))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["mesorectal_fascia_involement"]["margin"] = "missing"
            processed_data[i]["mesorectal_fascia_involement"]["circumferential_location_of_shortest_distance"] = None
            processed_data[i]["mesorectal_fascia_involement"]["involved_by"] = "missing"
            processed_data[i]["mesorectal_fascia_involement"]["shortest_distance_to_MRF_mm"] = None

    eligible_indices = []
    for i, record in enumerate(processed_data):
        margin = record["mesorectal_fascia_involement"]["margin"]
        if margin == "involved(<= 1 mm)":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.03))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["mesorectal_fascia_involement"]["circumferential_location_of_shortest_distance"] = None


    eligible_indices = []
    for i, record in enumerate(processed_data):
        by_field = record["mesorectal_fascia_involement"]["involved_by"]
        if by_field != "not applicable" and by_field != "missing":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.03))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            if processed_data[i]["mesorectal_fascia_involement"]["involved_by"] == "EMVI":
                processed_data[i]["emvi"]["presence_of_emvi"] = "missing"
                processed_data[i]["emvi"]["emvi_clock_position"] = None
            processed_data[i]["mesorectal_fascia_involement"]["involved_by"] = "missing"

    lymph_node_fields = [
        "regional_total_suspicious_mesorectal_nodes",
        "regional_total_suspicious_extramesorectal_nodes",
        "tumor_deposits_within_mesorectum",
        "nodes_short_axis_ge_9mm",
        "nodes_from_5_to_8mm_with_2_morphologic_criteria",
        "nodes_5mm_with_all_3_morphologic_criteria",
        "mucinous_nodes_any_size"
    ]

    # Replace zero counts with null and remove the unused location text field.
    for record in processed_data:
        lymph_data = record["lymph_nodes_and_tumor_deposits"]
        for field in lymph_node_fields:
            if lymph_data[field] == 0:
                lymph_data[field] = None
        lymph_data.pop("extramesorectal_location", None)


    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["lymph_nodes_and_tumor_deposits"]["n_stage"] = "missing"

    indices = get_random_indices(3, total_records)
    for i in indices:
        processed_data[i]["emvi"]["presence_of_emvi"] = "missing"
        processed_data[i]["emvi"]["emvi_clock_position"] = None

    eligible_indices = []
    for i, record in enumerate(processed_data):
        presence = record["emvi"]["presence_of_emvi"]
        if presence == "yes":
            eligible_indices.append(i)

    if eligible_indices:
        count = max(1, int(len(eligible_indices) * 0.05))
        selected = rng.sample(eligible_indices, min(count, len(eligible_indices)))
        for i in selected:
            processed_data[i]["emvi"]["emvi_clock_position"] = None

    return processed_data


def load_and_process_jsonl(file_path):
    """
    Load JSONL file, process it with missing data, and check distributions.

    Args:
        file_path (str): Path to the JSONL file

    Returns:
        tuple: (original_data, processed_data, original_distributions, processed_distributions)
    """

    all_outputs = []
    try:
        with Path(file_path).open(encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                try:
                    all_outputs.append(json.loads(line.strip()))
                except json.JSONDecodeError as e:
                    print(f"Error parsing JSON on line {line_num}: {line.strip()} - {e}")
                    continue
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please check the path.")
        return None, None, None, None
    except Exception as e:
        print(f"An unexpected error occurred while reading the file: {e}")
        return None, None, None, None

    print(f"Successfully loaded {len(all_outputs)} records from {file_path}")
    print("\nCalculating original distributions...")
    original_distributions = get_value_distributions(all_outputs)

    print("Processing data with missing values...")
    processed_data = process_medical_data(all_outputs)
    print("Calculating processed distributions...")
    processed_distributions = get_value_distributions(processed_data)

    return all_outputs, processed_data, original_distributions, processed_distributions


def print_distributions(distributions, title="Value Distributions"):
    """
    Print distributions in a formatted way.

    Args:
        distributions (dict): Distribution dictionary from get_value_distributions
        title (str): Title to print
    """
    print("\n" + "="*50)
    print(f"{title}:")
    print("="*50 + "\n")

    for field_path, value_counts in sorted(distributions.items()):
        print(f"Field: {field_path}")
        total_occurrences = sum(value_counts.values())
        for value, count in value_counts.items():
            percentage = (count / total_occurrences) * 100
            print(f"  '{value}': {count} occurrences ({percentage:.2f}%)")
        print("\n")
        print("-------------------------------------------------------------------")


def compare_distributions(original_dist, processed_dist):
    """
    Compare original and processed distributions to show changes.

    Args:
        original_dist (dict): Original distributions
        processed_dist (dict): Processed distributions
    """
    print("\n" + "="*60)
    print("COMPARISON: Changes After Processing")
    print("="*60 + "\n")

    all_fields = set(original_dist.keys()) | set(processed_dist.keys())

    for field in sorted(all_fields):
        orig_counts = original_dist.get(field, {})
        proc_counts = processed_dist.get(field, {})

        # check for differences
        all_values = set(orig_counts.keys()) | set(proc_counts.keys())
        has_changes = False

        for value in all_values:
            orig_count = orig_counts.get(value, 0)
            proc_count = proc_counts.get(value, 0)
            if orig_count != proc_count:
                has_changes = True
                break

        if has_changes:
            print(f"Field: {field}")
            print("BEFORE -> AFTER")

            for value in sorted(all_values):
                orig_count = orig_counts.get(value, 0)
                proc_count = proc_counts.get(value, 0)

                if orig_count != proc_count:
                    orig_total = sum(orig_counts.values()) if orig_counts else 0
                    proc_total = sum(proc_counts.values()) if proc_counts else 0

                    orig_pct = (orig_count / orig_total * 100) if orig_total > 0 else 0
                    proc_pct = (proc_count / proc_total * 100) if proc_total > 0 else 0

                    change = proc_count - orig_count
                    change_sign = "+" if change > 0 else ""

                    print(f" '{value}': {orig_count} ({orig_pct:.1f}%) -> {proc_count} ({proc_pct:.1f}%) [{change_sign}{change}]")

            print("-------------------------------------------------------------------\n")


In [5]:
original_data, processed_data, original_dist, processed_dist = load_and_process_jsonl(
    BALANCED_DATA_PATH
)
compare_distributions(original_dist, processed_dist)

with PROCESSED_DATA_PATH.open("w", encoding="utf-8") as f:
    for record in processed_data:
        f.write(json.dumps(record) + "\n")

print(f"\nProcessed data saved to: {PROCESSED_DATA_PATH}")
print(f"Total records processed: {len(processed_data)}")


Successfully loaded 1000 records from C:\Users\simmu\Desktop\Paper\synthetic_dataset\code_to_obtain_SR_data\jsonl_data\balanced_synthetic_SR.jsonl

Calculating original distributions...
Processing data with missing values...
Calculating processed distributions...

COMPARISON: Changes After Processing

Field: emvi.emvi_clock_position
BEFORE -> AFTER
 '1': 48 (4.8%) -> 42 (4.2%) [-6]
 '10': 46 (4.6%) -> 42 (4.2%) [-4]
 '12': 39 (3.9%) -> 36 (3.6%) [-3]
 '2': 34 (3.4%) -> 31 (3.1%) [-3]
 '3': 38 (3.8%) -> 36 (3.6%) [-2]
 '4': 44 (4.4%) -> 42 (4.2%) [-2]
 '5': 38 (3.8%) -> 34 (3.4%) [-4]
 '6': 39 (3.9%) -> 36 (3.6%) [-3]
 '7': 30 (3.0%) -> 25 (2.5%) [-5]
 '8': 32 (3.2%) -> 28 (2.8%) [-4]
 '9': 23 (2.3%) -> 19 (1.9%) [-4]
 'None': 538 (53.8%) -> 578 (57.8%) [+40]
-------------------------------------------------------------------

Field: emvi.presence_of_emvi
BEFORE -> AFTER
 'missing': 0 (0.0%) -> 32 (3.2%) [+32]
 'no': 538 (53.8%) -> 524 (52.4%) [-14]
 'yes': 462 (46.2%) -> 444 (44.4%) [-

### Check the final distributions

In [6]:
# Reuse the processed records already in memory instead of reloading the file.
all_outputs = processed_data
print(f"Analysing {len(all_outputs)} processed records")

distributions = get_value_distributions(all_outputs)
print("\n" + "-" * 50)
print("Value Distributions Across All Fields:")
print("-" * 50 + "\n")
for field_path, value_counts in sorted(distributions.items()):
    print(f"Field: {field_path}")
    total_occurrences = sum(value_counts.values())
    for value, count in value_counts.items():
        percentage = count / total_occurrences * 100
        print(f"  '{value}': {count} occurrences ({percentage:.2f}%)")
    print("\n")
    print("-" * 67)


Analysing 1000 processed records

--------------------------------------------------
Value Distributions Across All Fields:
--------------------------------------------------

Field: emvi.emvi_clock_position
  'None': 578 occurrences (57.80%)
  '2': 31 occurrences (3.10%)
  '3': 36 occurrences (3.60%)
  '8': 28 occurrences (2.80%)
  '10': 42 occurrences (4.20%)
  '11': 51 occurrences (5.10%)
  '7': 25 occurrences (2.50%)
  '5': 34 occurrences (3.40%)
  '4': 42 occurrences (4.20%)
  '9': 19 occurrences (1.90%)
  '6': 36 occurrences (3.60%)
  '1': 42 occurrences (4.20%)
  '12': 36 occurrences (3.60%)


-------------------------------------------------------------------
Field: emvi.presence_of_emvi
  'no': 524 occurrences (52.40%)
  'yes': 444 occurrences (44.40%)
  'missing': 32 occurrences (3.20%)


-------------------------------------------------------------------
Field: local_tumor_status.circumferential_tumor_involvement_end
  '7': 60 occurrences (6.00%)
  '5': 81 occurrences (8.10%

### We will randomly modify values in the fields with measurements (mm to cm and vice versa)

In [7]:
conversion_rng = random.Random(UNIT_CONVERSION_SEED)


def convert_and_replace(field_name, value):
    if isinstance(value, (int, float)) and conversion_rng.random() < 0.4:
        return field_name.replace("_cm", "_mm"), int(value * 10)
    return field_name, value


with PROCESSED_DATA_PATH.open("r", encoding="utf-8") as infile, MODIFIED_DATA_PATH.open(
    "w", encoding="utf-8"
) as outfile:
    for line in infile:
        record = json.loads(line, object_pairs_hook=OrderedDict)
        local_status = record.get("local_tumor_status", OrderedDict())
        new_local_status = OrderedDict()

        for key, value in local_status.items():
            if key in {
                "distance_from_anorectal_junction_to_distal_tumor_border_cm",
                "tumor_length_cm",
            }:
                new_key, new_value = convert_and_replace(key, value)
            else:
                new_key, new_value = key, value
            new_local_status[new_key] = new_value

        record["local_tumor_status"] = new_local_status
        outfile.write(json.dumps(record) + "\n")


In [8]:
with MODIFIED_DATA_PATH.open(encoding="utf-8") as f:
    for index, line in enumerate(f):
        if index >= 100:
            break
        print(json.dumps(json.loads(line), indent=2))
        print("\n" + "=" * 50 + "\n")


{
  "local_tumor_status": {
    "morphology": "polypoid",
    "mucinous_component": "no",
    "circumferential_tumor_involvement_start": 5,
    "circumferential_tumor_involvement_end": 7,
    "rectal_tumor_location_in_rectum": "low",
    "distance_from_anorectal_junction_to_distal_tumor_border_mm": 11,
    "relation_to_anterior_peritoneal_reflection": "below",
    "tumor_length_cm": 5.1,
    "t_stage": "T3b",
    "invaded_structures": "",
    "sphincter_complex_invasion": "no",
    "lowest_part_of_sphincter_invasion": "not applicable"
  },
  "mesorectal_fascia_involement": {
    "shortest_distance_to_MRF_mm": 6.2,
    "margin": "clear(> 1 mm)",
    "circumferential_location_of_shortest_distance": null,
    "involved_by": "not applicable"
  },
  "lymph_nodes_and_tumor_deposits": {
    "regional_total_suspicious_mesorectal_nodes": 1,
    "nodes_short_axis_ge_9mm": null,
    "nodes_from_5_to_8mm_with_2_morphologic_criteria": 1,
    "nodes_5mm_with_all_3_morphologic_criteria": null,
    "m